In [ ]:
import osmnx as ox
from osmnx.features import features_from_bbox
import geopandas as gpd
from src.feature_building_utils import *
from src.geometric_utils import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import toml

In [ ]:
df = pd.read_parquet('data/processed_data/S3-approx-coordinates.parquet')

In [ ]:
docs = toml.load("documentation/feature_docs.toml")

In [ ]:
tags_amenity = {"amenity": True}
gdf_amenity = features_from_bbox(BBOX, tags_amenity)

In [ ]:
for col in gdf_amenity.columns:
    print(col)
    print(len(gdf_amenity[~gdf_amenity[col].isna()]))
    print(gdf_amenity[~gdf_amenity[col].isna()][col].unique())
    print("==="*20)

In [ ]:
feature_name = "close2smoking_amenity_30"

# build your “smoking‐allowed” subset first
smoking_allowed = gdf_amenity[
    (gdf_amenity['outdoor_seating'] == "yes") |
    (gdf_amenity['smoking'].isin(["yes", "outside"]))
]

# then apply
df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=smoking_allowed,
            point=Point(row.x, row.y),
            threshold=30
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 30m away from an amenity where smoking is usually allowed."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2cigarette_disposal_50"

df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=gdf_amenity,
            point=Point(row.x, row.y),
            threshold=50,
            type_column="waste",
            types=["cigarettes;mixed", "cigarettes"]
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 50m away from a waste basket where cigarettes are usually disposed of."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2waste_disposal_25"

df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=gdf_amenity,
            point=Point(row.x, row.y),
            threshold=25,
            type_column="amenity",
            types=['waste_basket', 'recycling', 'waste_disposal']
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 25m away from a waste basket."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"


In [ ]:
feature_name = "close2parking_30"

# build your “parking‐space” subset first
parking_space = gdf_amenity[
    (gdf_amenity['amenity'].isin(['parking', 'parking_space', 'parking_entrance', "motorcycle_parking"])) |
    (gdf_amenity['parking'].notna())
]

df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=parking_space,
            point=Point(row.x, row.y),
            threshold=30,
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 30m away from a parking space."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2fuel_station_40"

df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=gdf_amenity,
            point=Point(row.x, row.y),
            threshold=40,
            type_column="amenity",
            types=['fuel']
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 40m away from a fuel station."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2drinking_water_20"

df[feature_name] = (
    df.apply(
        lambda row: is_close_to(
            features=gdf_amenity,
            point=Point(row.x, row.y),
            threshold=30,
            type_column="amenity",
            types=['drinking_water', 'fountain']
        ),
        axis=1
    )
    .astype(int)
)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 20m away from a drinking water source."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_amenity.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
df[feature_name].value_counts()

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')